In [1]:
"""
NLTK 文本分析工作台
基于 NLTK 入门笔记的五个核心功能：
  1. 词频分析 (FreqDist)
  2. 词汇多样性
  3. 词汇筛选 (列表推导)
  4. Concordance 上下文检索
  5. 双连词 (Bigrams)
"""

import nltk
from nltk import FreqDist, bigrams
from nltk.tokenize import word_tokenize

# 首次运行需要下载资源
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── 示例文本（可替换为任意英文文本）──────────────────────────────────────────
SAMPLE_TEXT = """
Call me Ishmael. Some years ago never mind how long precisely having little money
in my purse and nothing particular to interest me on shore I thought I would sail
about a little and see the watery part of the world. It is a way I have of driving
off the spleen and regulating the circulation. Whenever I find myself growing grim
about the mouth whenever it is a damp drizzly November in my soul whenever I find
myself involuntarily pausing before coffin warehouses and bringing up the rear of
every funeral I meet and especially whenever my hypos get such an upper hand of me
that it requires a strong moral principle to prevent me from deliberately stepping
into the street and methodically knocking people hats off then I account it high
time to get to sea as soon as I can.
"""


# ── 工具函数 ─────────────────────────────────────────────────────────────────

def load_tokens(text: str) -> list[str]:
    """分词，返回原始 token 列表（保留大小写）"""
    return word_tokenize(text)


def sep(title: str):
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print('─'*55)


# ── 1. 词频分析 ───────────────────────────────────────────────────────────────

def freq_analysis(tokens: list[str], top_n: int = 15):
    sep(f"词频分析  Top {top_n}")
    # 只统计纯字母词，忽略大小写
    words = [w.lower() for w in tokens if w.isalpha()]
    fdist = FreqDist(words)

    print(f"{'词':>15}  {'次数':>5}  {'占比':>7}  条形图")
    total = len(words)
    max_count = fdist.most_common(1)[0][1]
    for word, count in fdist.most_common(top_n):
        bar = '█' * int(count / max_count * 20)
        pct = 100 * count / total
        print(f"{word:>15}  {count:>5}  {pct:>6.2f}%  {bar}")
    return fdist


# ── 2. 词汇多样性 ─────────────────────────────────────────────────────────────

def lexical_diversity(tokens: list[str]):
    sep("词汇多样性")
    total = len(tokens)
    unique = len(set(tokens))
    # 只计纯字母、不区分大小写
    alpha_lower = [w.lower() for w in tokens if w.isalpha()]
    unique_alpha = len(set(alpha_lower))
    diversity = unique / total
    avg_use = total / unique

    print(f"  总标识符 (len):                 {total}")
    print(f"  词类型   (len set):             {unique}")
    print(f"  纯字母词类型 (小写去重):        {unique_alpha}")
    print(f"  词汇丰富度 len/len(set):        {diversity:.4f}")
    print(f"  每词平均使用次数:               {avg_use:.2f}")
    if diversity > 0.6:
        print("  → 词汇较丰富")
    else:
        print("  → 存在大量重复词")


# ── 3. 词汇筛选（列表推导） ───────────────────────────────────────────────────

def word_filter(tokens: list[str], fdist: FreqDist):
    sep("词汇筛选 (列表推导)")
    words_lower = [w.lower() for w in tokens if w.isalpha()]
    vocab = set(words_lower)

    # 长词且高频：len > 6 且出现次数 > 1
    long_freq = sorted([w for w in vocab if len(w) > 6 and fdist[w] > 1])
    print(f"\n  [长度>6 且 频次>1]  共 {len(long_freq)} 个:")
    print("  ", long_freq)

    # 首字母大写词（原始 tokens）
    title_words = sorted(set([w for w in tokens if w.istitle()]))
    print(f"\n  [首字母大写 istitle()]  共 {len(title_words)} 个:")
    print("  ", title_words)

    # Hapaxes：只出现一次的词
    hapaxes = sorted(fdist.hapaxes())
    print(f"\n  [Hapaxes - 只出现1次]  共 {len(hapaxes)} 个:")
    print("  ", hapaxes[:20], "..." if len(hapaxes) > 20 else "")

    # 包含某子串
    substr = "ing"
    ing_words = sorted([w for w in vocab if substr in w])
    print(f"\n  [包含子串 '{substr}']  共 {len(ing_words)} 个:")
    print("  ", ing_words)


# ── 4. Concordance（上下文检索） ─────────────────────────────────────────────

def concordance(tokens: list[str], target: str, window: int = 5):
    sep(f"Concordance: '{target}'  (窗口 ±{window})")
    results = []
    for i, w in enumerate(tokens):
        if w.lower() == target.lower():
            left = tokens[max(0, i - window):i]
            right = tokens[i + 1:i + 1 + window]
            results.append((left, w, right))

    if not results:
        print(f"  未找到词 '{target}'")
        return

    print(f"  共出现 {len(results)} 次\n")
    for left, w, right in results:
        l_str = ' '.join(left).rjust(35)
        r_str = ' '.join(right)
        print(f"  {l_str}  [{w}]  {r_str}")


# ── 5. 双连词（Bigrams） ──────────────────────────────────────────────────────

def bigram_analysis(tokens: list[str], top_n: int = 10):
    sep(f"双连词 (Bigrams)  Top {top_n}")
    words = [w.lower() for w in tokens if w.isalpha()]
    bg = list(bigrams(words))
    fdist_bg = FreqDist(bg)

    print(f"  共 {len(bg)} 个双连词，{len(set(bg))} 个唯一双连词\n")
    print(f"  {'双连词':<25}  {'次数':>5}")
    for pair, count in fdist_bg.most_common(top_n):
        print(f"  {pair[0]+' '+pair[1]:<25}  {count:>5}")


# ── 主程序 ────────────────────────────────────────────────────────────────────

def main():
    print("=" * 55)
    print("  NLTK 文本分析工作台")
    print("=" * 55)

    # 可以替换成 input() 让用户粘贴文本
    text = SAMPLE_TEXT

    tokens = load_tokens(text)
    print(f"\n  文本已加载，共 {len(tokens)} 个 token")

    fdist = freq_analysis(tokens, top_n=15)
    lexical_diversity(tokens)
    word_filter(tokens, fdist)
    concordance(tokens, target="find", window=5)
    concordance(tokens, target="I", window=4)
    bigram_analysis(tokens, top_n=10)

    print("\n" + "=" * 55)
    print("  分析完成")
    print("=" * 55)


if __name__ == "__main__":
    main()

  NLTK 文本分析工作台

  文本已加载，共 147 个 token

───────────────────────────────────────────────────────
  词频分析  Top 15
───────────────────────────────────────────────────────
              词     次数       占比  条形图
              i      8    5.59%  ████████████████████
            the      7    4.90%  █████████████████
            and      6    4.20%  ███████████████
             me      4    2.80%  ██████████
             to      4    2.80%  ██████████
              a      4    2.80%  ██████████
             of      4    2.80%  ██████████
             it      4    2.80%  ██████████
       whenever      4    2.80%  ██████████
             my      3    2.10%  ███████
         little      2    1.40%  █████
             in      2    1.40%  █████
          about      2    1.40%  █████
             is      2    1.40%  █████
            off      2    1.40%  █████

───────────────────────────────────────────────────────
  词汇多样性
───────────────────────────────────────────────────────
  总标识符 (len):         